# Learning Effectiveness ML Pipeline - Exploratory Data Analysis

**Project:** ASTAR Learning Method Effectiveness Prediction  
**Data Source:** Canvas LMS + ASTAR Session Behavioral Data  
**Goal:** Understand feature distributions, correlations, and patterns before modeling

## Notebook Contents:
1. Data Loading & Overview
2. Missing Value Analysis
3. Feature Distributions
4. Correlation Analysis
5. Outcome Variable Analysis
6. Behavioral Pattern Exploration
7. Key Insights & Next Steps


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✅ Libraries imported successfully")


## 1. Data Loading & Overview


In [ ]:
# Load training data
data_path = Path('../../data/training_sample.csv')
df = pd.read_csv(data_path)

print(f"📊 Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\n📋 First 5 rows:")
df.head()


In [ ]:
# Data types and info
print("📝 Column Data Types:")
df.info()


In [ ]:
# Summary statistics
print("📈 Numerical Feature Statistics:")
df.describe().T


## 2. Missing Value Analysis


In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
}).sort_values('Missing Count', ascending=False)

print("🔍 Missing Values:")
print(missing_df[missing_df['Missing Count'] > 0])

# Visualize missing values
plt.figure(figsize=(10, 6))
missing_cols = missing_df[missing_df['Missing Count'] > 0]
if len(missing_cols) > 0:
    plt.barh(missing_cols.index, missing_cols['Percentage'])
    plt.xlabel('Missing Percentage (%)')
    plt.title('Missing Values by Feature')
    plt.tight_layout()
    plt.savefig('../../ml/reports/missing_values.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("✅ No missing values found!")


## 3. Feature Distributions


In [ ]:
# Key behavioral features
behavioral_features = [
    'num_sessions', 'num_messages', 'avg_messages_per_session',
    'used_step_mode', 'num_concepts', 'num_context_items',
    'session_duration_hours', 'engagement_score'
]

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.ravel()

for idx, feature in enumerate(behavioral_features):
    if feature in df.columns:
        axes[idx].hist(df[feature].dropna(), bins=20, edgecolor='black', alpha=0.7)
        axes[idx].set_title(f'{feature}')
        axes[idx].set_xlabel('Value')
        axes[idx].set_ylabel('Frequency')
        axes[idx].grid(True, alpha=0.3)

# Hide unused subplots
for idx in range(len(behavioral_features), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('../../ml/reports/feature_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Feature distributions plotted")


## 4. Correlation Analysis


In [ ]:
# Select numeric columns for correlation
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# Remove ID columns
numeric_cols = [col for col in numeric_cols if 'id' not in col.lower()]

# Compute correlation matrix
corr_matrix = df[numeric_cols].corr()

# Plot heatmap
plt.figure(figsize=(14, 12))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Heatmap', fontsize=16, pad=20)
plt.tight_layout()
plt.savefig('../../ml/reports/correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

# Top correlations with is_completed
if 'is_completed' in corr_matrix.columns:
    print("\n📊 Top Correlations with Assignment Completion:")
    completion_corr = corr_matrix['is_completed'].sort_values(ascending=False)
    print(completion_corr[completion_corr.index != 'is_completed'])


## 5. Outcome Variable Analysis


In [ ]:
# Assignment completion rate
completion_rate = df['is_completed'].mean() * 100
print(f"📈 Overall Completion Rate: {completion_rate:.1f}%")

# Completion by grade bucket
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Completion distribution
df['is_completed'].value_counts().plot(kind='bar', ax=axes[0], color=['#e74c3c', '#2ecc71'])
axes[0].set_title('Assignment Completion Distribution')
axes[0].set_xlabel('Completed (1=Yes, 0=No)')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['Not Completed', 'Completed'], rotation=0)

# Grade bucket distribution (excluding Unknown)
grade_counts = df[df['grade_bucket'] != 'Unknown']['grade_bucket'].value_counts()
grade_counts.plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Grade Distribution (Completed Assignments)')
axes[1].set_xlabel('Grade Bucket')
axes[1].set_ylabel('Count')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45)

plt.tight_layout()
plt.savefig('../../ml/reports/outcome_analysis.png', dpi=300, bbox_inches='tight')
plt.show()


## 6. Key Insights

**Data Quality:**
- Total samples: {df.shape[0]}
- Features: {df.shape[1]}
- Completion rate: {completion_rate:.1f}%

**Next Steps:**
1. **Clustering (Notebook 02):** Identify learner archetypes using K-Means, GMM, DBSCAN
2. **Classification (Notebook 03):** Build predictive models for assignment completion
3. **Feature Engineering:** Consider interaction terms (e.g., engagement_score × days_until_due)
4. **Model Evaluation:** Focus on precision/recall for minority class (non-completion)
